In [4]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score
import joblib
df = pd.read_csv(r"C:\Users\Admin\Desktop\-rainfall-pradction-ml\eth_householdgeovariables_y5.csv")


ImportError: Unable to import required dependencies:
numpy: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [4]:
df.head()

,household_id,dist_road,dist_market,dist_border,dist_popcenter,dist_admhq,af_bio_1_x,af_bio_8_x,af_bio_12_x,af_bio_13_x,...,c2_evimax_avg,c2_grn_avg,c2_sen_avg,c2_h2021_eviarea,c2_h2021_evimax,c2_h2021_grn,c2_h2021_sen,lat_dd_mod,lon_dd_mod,suppress
0,20101010100104011,7.7,162.300003,82.900002,0.4,0.0,283,307,184,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.749938,41.080363,0
1,20101010100104022,7.7,162.300003,82.900002,0.4,0.0,283,307,184,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.749938,41.080363,0
2,20101010100104033,7.7,162.300003,82.900002,0.4,0.0,283,307,184,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.749938,41.080363,0
3,20101010100104044,7.7,162.300003,82.900002,0.4,0.0,283,307,184,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.749938,41.080363,0
4,20101010100104055,7.7,162.300003,82.900002,0.4,0.0,283,307,184,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.749938,41.080363,0


In [5]:
print(df.shape)

(4890, 52)


In [6]:
print(df.isnull().sum())

household_id            0
dist_road               0
dist_market             0
dist_border             0
dist_popcenter          0
dist_admhq              0
af_bio_1_x              0
af_bio_8_x              0
af_bio_12_x             0
af_bio_13_x             0
af_bio_16_x             0
afmnslp_pct             0
srtm_1k                 0
popdensity              0
cropshare               0
sq1                     0
sq2                     0
sq3                     0
sq4                     0
sq5                     0
sq6                     0
sq7                     0
ssa_aez09               0
landcov                 0
twi_ne                  0
pct_urban_cluster       0
pct_urban_center        0
h2021_tot               0
h2021_wetQstart         0
h2021_wetQ              0
anntot_avg              0
wetQ_avgstart           0
wetQ_avg                0
eviarea_avg             0
evimax_avg              0
grn_avg                 0
sen_avg                 0
h2021_eviarea           0
h2021_evimax

In [7]:
# Drop unnecessary columns
cols_to_drop = [
    "household_id",       # ID, not useful
    "suppress",           # control flag
    "af_bio_12_x",        # target variable (keep separately)
    # Columns with lots of missing values
    "c2_eviarea_avg", "c2_evimax_avg", "c2_grn_avg", "c2_sen_avg",
    "c2_h2021_eviarea", "c2_h2021_evimax", "c2_h2021_grn", "c2_h2021_sen",
    # Optional low-importance columns (can drop after feature importance check)
    "sq1", "sq2", "sq3", "sq4", "sq5", "sq6", "sq7"
]

df_clean = df.drop(columns=cols_to_drop, errors="ignore")

# Step 3: Define target and features
y = df["af_bio_12_x"]  # rainfall
X = df_clean.copy()    # all remaining columns are features

# Step 4: Fill missing values in features
X = X.fillna(X.mean(numeric_only=True))

print(" Cleaned dataset shape:", X.shape)
print("Features:", X.columns)

 Cleaned dataset shape: (4890, 34)
Features: Index(['dist_road', 'dist_market', 'dist_border', 'dist_popcenter',
       'dist_admhq', 'af_bio_1_x', 'af_bio_8_x', 'af_bio_13_x', 'af_bio_16_x',
       'afmnslp_pct', 'srtm_1k', 'popdensity', 'cropshare', 'ssa_aez09',
       'landcov', 'twi_ne', 'pct_urban_cluster', 'pct_urban_center',
       'h2021_tot', 'h2021_wetQstart', 'h2021_wetQ', 'anntot_avg',
       'wetQ_avgstart', 'wetQ_avg', 'eviarea_avg', 'evimax_avg', 'grn_avg',
       'sen_avg', 'h2021_eviarea', 'h2021_evimax', 'h2021_grn', 'h2021_sen',
       'lat_dd_mod', 'lon_dd_mod'],
      dtype='object')


In [8]:
cols_to_keep = [
    'dist_road', 'dist_market', 'dist_border', 'dist_popcenter',
    'dist_admhq', 'af_bio_1_x', 'af_bio_8_x', 'af_bio_13_x', 'af_bio_16_x',
    'afmnslp_pct', 'srtm_1k', 'popdensity', 'cropshare', 'ssa_aez09',
    'landcov', 'twi_ne', 'pct_urban_cluster', 'pct_urban_center',
    'h2021_tot', 'h2021_wetQstart', 'h2021_wetQ', 'anntot_avg',
    'wetQ_avgstart', 'wetQ_avg', 'eviarea_avg', 'evimax_avg', 'grn_avg',
    'sen_avg', 'h2021_eviarea', 'h2021_evimax', 'h2021_grn', 'h2021_sen',
    'lat_dd_mod', 'lon_dd_mod'
]

df_features = df[cols_to_keep].copy()

# Fill missing numeric values with column mean
df_features = df_features.fillna(df_features.mean(numeric_only=True))

# Fill missing categorical variables with mode (if any)
for col in df_features.select_dtypes(include='object').columns:
    df_features[col] = df_features[col].fillna(df_features[col].mode()[0])

print(" Missing values filled")
print(df_features.isnull().sum()) 

 Missing values filled
dist_road            0
dist_market          0
dist_border          0
dist_popcenter       0
dist_admhq           0
af_bio_1_x           0
af_bio_8_x           0
af_bio_13_x          0
af_bio_16_x          0
afmnslp_pct          0
srtm_1k              0
popdensity           0
cropshare            0
ssa_aez09            0
landcov              0
twi_ne               0
pct_urban_cluster    0
pct_urban_center     0
h2021_tot            0
h2021_wetQstart      0
h2021_wetQ           0
anntot_avg           0
wetQ_avgstart        0
wetQ_avg             0
eviarea_avg          0
evimax_avg           0
grn_avg              0
sen_avg              0
h2021_eviarea        0
h2021_evimax         0
h2021_grn            0
h2021_sen            0
lat_dd_mod           0
lon_dd_mod           0
dtype: int64


In [9]:

# Step 1: Add target variable as a new column
df_cleaned = df_features.copy()
df_cleaned['rainfall'] = df['af_bio_12_x']

# Step 2: Check the new dataframe
print("Combined dataset shape:", df_cleaned.shape)
print(df_cleaned.head())




Combined dataset shape: (4890, 35)
   dist_road  dist_market  dist_border  dist_popcenter  dist_admhq  \
0        7.7   162.300003    82.900002             0.4         0.0   
1        7.7   162.300003    82.900002             0.4         0.0   
2        7.7   162.300003    82.900002             0.4         0.0   
3        7.7   162.300003    82.900002             0.4         0.0   
4        7.7   162.300003    82.900002             0.4         0.0   

   af_bio_1_x  af_bio_8_x  af_bio_13_x  af_bio_16_x  afmnslp_pct  ...  \
0         283         307           47           98            2  ...   
1         283         307           47           98            2  ...   
2         283         307           47           98            2  ...   
3         283         307           47           98            2  ...   
4         283         307           47           98            2  ...   

   evimax_avg  grn_avg  sen_avg h2021_eviarea h2021_evimax  h2021_grn  \
0       0.342      187      280 

In [10]:
# Step 3: Check for duplicate rows
duplicates = df_cleaned.duplicated()

# Print number of duplicate rows
print("Number of duplicate rows:", duplicates.sum())

# Optional: view duplicate rows
print("\nDuplicate rows:")
print(df_cleaned[duplicates])


Number of duplicate rows: 4376

Duplicate rows:
      dist_road  dist_market  dist_border  dist_popcenter  dist_admhq  \
1           7.7   162.300003    82.900002             0.4         0.0   
2           7.7   162.300003    82.900002             0.4         0.0   
3           7.7   162.300003    82.900002             0.4         0.0   
4           7.7   162.300003    82.900002             0.4         0.0   
5           7.7   162.300003    82.900002             0.4         0.0   
...         ...          ...          ...             ...         ...   
4885        8.3    22.400000   183.899994            22.4         0.2   
4886        8.3    22.400000   183.899994            22.4         0.2   
4887        8.3    22.400000   183.899994            22.4         0.2   
4888        8.3    22.400000   183.899994            22.4         0.2   
4889        8.3    22.400000   183.899994            22.4         0.2   

      af_bio_1_x  af_bio_8_x  af_bio_13_x  af_bio_16_x  afmnslp_pct  ...  \

In [11]:
# Step 4: Remove duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# Check the shape after removing duplicates
print(" Dataset shape after removing duplicates:", df_cleaned.shape)

# Verify no more duplicates
print("Number of duplicates now:", df_cleaned.duplicated().sum())


 Dataset shape after removing duplicates: (514, 35)
Number of duplicates now: 0


In [12]:
numeric_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns

# Compute Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df_cleaned[numeric_cols].quantile(0.25)
Q3 = df_cleaned[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# Find outliers per column
outliers = pd.DataFrame(False, index=df_cleaned.index, columns=numeric_cols)

for col in numeric_cols:
    lower = Q1[col] - 1.5 * IQR[col]
    upper = Q3[col] + 1.5 * IQR[col]
    outliers[col] = (df_cleaned[col] < lower) | (df_cleaned[col] > upper)

# Count outliers per column
print("\nNumber of outliers per column:")
print(outliers.sum())

# View rows that contain any outlier
rows_with_outliers = df_cleaned[outliers.any(axis=1)]
print(f"\nTotal rows containing any outlier: {len(rows_with_outliers)}")
print("\nRows with any outlier detected:")
print(rows_with_outliers.head(20))  #


Number of outliers per column:
dist_road            62
dist_market          22
dist_border           0
dist_popcenter       31
dist_admhq            2
af_bio_1_x            0
af_bio_8_x            0
af_bio_13_x           0
af_bio_16_x           0
afmnslp_pct          14
srtm_1k               0
popdensity           51
cropshare             0
twi_ne               22
pct_urban_cluster    26
pct_urban_center      0
h2021_tot            17
h2021_wetQstart       0
h2021_wetQ           44
anntot_avg            0
wetQ_avgstart         0
wetQ_avg              9
eviarea_avg           0
evimax_avg            0
grn_avg               0
sen_avg              44
h2021_eviarea         0
h2021_evimax          0
h2021_grn             0
h2021_sen            63
lat_dd_mod           27
lon_dd_mod            0
rainfall              3
dtype: int64

Total rows containing any outlier: 234

Rows with any outlier detected:
     dist_road  dist_market  dist_border  dist_popcenter  dist_admhq  \
14    9.300000   1

In [13]:
numeric_cols = df_cleaned.select_dtypes(include=['number']).columns

Q1 = df_cleaned[numeric_cols].quantile(0.25)
Q3 = df_cleaned[numeric_cols].quantile(0.75)


IQR = Q3 - Q1

# Clip each numeric column based on IQR thresholds
for col in numeric_cols:
    lower = Q1[col] - 1.5 * IQR[col]
    upper = Q3[col] + 1.5 * IQR[col]
    df_cleaned[col] = df_cleaned[col].clip(lower, upper)


In [14]:
# After capping, check min and max values for numeric columns
print(df_cleaned[numeric_cols].agg(['min', 'max']))


df_cleaned.head()

     dist_road  dist_market  dist_border  dist_popcenter  dist_admhq  \
min        0.0     0.400000          3.2        0.400000         0.0   
max       21.0   209.149996        531.5       89.187498         0.5   

     af_bio_1_x  af_bio_8_x  af_bio_13_x  af_bio_16_x  afmnslp_pct  ...  \
min          95          89           37           78            1  ...   
max         293         319          433         1103           29  ...   

     evimax_avg  grn_avg  sen_avg  h2021_eviarea  h2021_evimax  h2021_grn  \
min       0.269       97    257.5          4.190         0.258         94   
max       0.641      187    289.0         35.444         0.666        174   

     h2021_sen  lat_dd_mod  lon_dd_mod  rainfall  
min      255.5    5.092198   33.434831   145.000  
max      299.5   12.333356   47.307840  1945.125  

[2 rows x 33 columns]


,dist_road,dist_market,dist_border,dist_popcenter,dist_admhq,af_bio_1_x,af_bio_8_x,af_bio_13_x,af_bio_16_x,afmnslp_pct,...,evimax_avg,grn_avg,sen_avg,h2021_eviarea,h2021_evimax,h2021_grn,h2021_sen,lat_dd_mod,lon_dd_mod,rainfall
0,7.7,162.300003,82.900002,0.400000,0.0,283,307,47,98,2,...,0.342,187,280.0,9.156000,0.343,174,275.0,11.749938,41.080363,184.0
14,9.3,169.800003,75.699997,8.500000,0.0,284,309,45,93,1,...,0.342,187,280.0,9.156000,0.343,174,275.0,11.707530,41.146505,174.0
24,0.5,209.149996,36.500000,74.900002,0.1,283,319,37,83,2,...,0.342,187,280.0,9.156000,0.343,174,275.0,12.057510,41.933473,163.0
35,1.3,194.000000,48.700001,5.600000,0.1,287,315,38,78,2,...,0.342,187,280.0,9.156000,0.343,174,275.0,11.562991,41.432114,145.0
36,0.7,9.800000,249.300003,9.800000,0.3,149,161,284,700,18,...,0.398,167,273.0,16.330999,0.408,145,274.0,11.133939,39.639801,1136.0


In [15]:
import numpy as np

# Identify numeric columns
numeric_cols = df_cleaned.select_dtypes(include=['number']).columns

# Apply log transformation (adding 1 to avoid log(0) issues)
for col in numeric_cols:
    # Only apply if all values are non-negative
    if (df_cleaned[col] >= 0).all():
        df_cleaned[col + '_log'] = np.log1p(df_cleaned[col])
    else:
        print(f"Column {col} has negative values, skipping log transformation.")

# Check the first few rows after transformation
df_cleaned.head()


,dist_road,dist_market,dist_border,dist_popcenter,dist_admhq,af_bio_1_x,af_bio_8_x,af_bio_13_x,af_bio_16_x,afmnslp_pct,...,evimax_avg_log,grn_avg_log,sen_avg_log,h2021_eviarea_log,h2021_evimax_log,h2021_grn_log,h2021_sen_log,lat_dd_mod_log,lon_dd_mod_log,rainfall_log
0,7.7,162.300003,82.900002,0.400000,0.0,283,307,47,98,2,...,0.294161,5.236442,5.638355,2.318065,0.294906,5.164786,5.620401,2.545526,3.739581,5.220356
14,9.3,169.800003,75.699997,8.500000,0.0,284,309,45,93,1,...,0.294161,5.236442,5.638355,2.318065,0.294906,5.164786,5.620401,2.542195,3.741152,5.164786
24,0.5,209.149996,36.500000,74.900002,0.1,283,319,37,83,2,...,0.294161,5.236442,5.638355,2.318065,0.294906,5.164786,5.620401,2.569363,3.759652,5.099866
35,1.3,194.000000,48.700001,5.600000,0.1,287,315,38,78,2,...,0.294161,5.236442,5.638355,2.318065,0.294906,5.164786,5.620401,2.530755,3.747905,4.983607
36,0.7,9.800000,249.300003,9.800000,0.3,149,161,284,700,18,...,0.335043,5.123964,5.613128,2.852497,0.342170,4.983607,5.616771,2.496006,3.704748,7.036148


In [16]:

X = df_cleaned.drop(columns=['rainfall'])
y = df_cleaned['rainfall']

# Identify categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", cat_cols)

# Only encode if there are categorical columns
if cat_cols:
    X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
else:
    X_encoded = X.copy()  # No encoding needed

# Combine with target
df_encoded = X_encoded.copy()
df_encoded['rainfall'] = y

print("Encoded dataset shape:", df_encoded.shape)
df_encoded.head()


Categorical columns to encode: ['ssa_aez09', 'landcov']
Encoded dataset shape: (514, 80)


,dist_road,dist_market,dist_border,dist_popcenter,dist_admhq,af_bio_1_x,af_bio_8_x,af_bio_13_x,af_bio_16_x,afmnslp_pct,...,ssa_aez09_Tropic-warm/subhumid,landcov_Built-up,landcov_Closed forest deciduous broad leaf,landcov_Closed forest evergreen broad leaf,landcov_Cropland,landcov_Herbaceous vegetation,landcov_Open forest deciduous broad leaf,landcov_Open forest unknown,landcov_Shrubs,rainfall
0,7.7,162.300003,82.900002,0.400000,0.0,283,307,47,98,2,...,False,False,False,False,False,False,False,False,False,184.0
14,9.3,169.800003,75.699997,8.500000,0.0,284,309,45,93,1,...,False,False,False,False,True,False,False,False,False,174.0
24,0.5,209.149996,36.500000,74.900002,0.1,283,319,37,83,2,...,False,False,False,False,False,False,False,False,False,163.0
35,1.3,194.000000,48.700001,5.600000,0.1,287,315,38,78,2,...,False,False,False,False,True,False,False,False,False,145.0
36,0.7,9.800000,249.300003,9.800000,0.3,149,161,284,700,18,...,False,True,False,False,False,False,False,False,False,1136.0


In [17]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Drop target column
X = df_encoded.drop(columns=['rainfall'])

# Ensure all columns are numeric (VIF only works for numeric)
X = X.select_dtypes(include=['number'])

# Create a DataFrame to store VIF values
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Sort by highest VIF first
vif_data = vif_data.sort_values(by="VIF", ascending=False)

vif_data.head(20)


,Feature,VIF
57,sen_avg_log,1.936065e+08
61,h2021_sen_log,1.888157e+08
63,lon_dd_mod_log,1.785366e+07
25,sen_avg,6.080809e+06
29,h2021_sen,5.887347e+06
45,twi_ne_log,2.309326e+06
55,evimax_avg_log,1.345175e+06
56,grn_avg_log,1.328820e+06
31,lon_dd_mod,1.232473e+06
60,h2021_grn_log,1.194987e+06


In [18]:
to_drop = [
    'sen_avg', 'h2021_sen',
    'af_bio_8_x', 'h2021_evimax', 'evimax_avg',
    'af_bio_1_x', 'lon_dd_mod', 'lat_dd_mod',
    'grn_avg', 'h2021_grn', 'eviarea_avg', 'af_bio_16_x', 'af_bio_13_x', 
    'anntot_avg', 'wetQ_avg', 
    'h2021_tot', 'h2021_wetQ', 
    'wetQ_avgstart', 'twi_ne'
]

X_reduced = X.drop(columns=to_drop)


In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Ensure numeric only
X_reduced = X_reduced.select_dtypes(include=['number'])

# Compute new VIF values
vif_data = pd.DataFrame()
vif_data["Feature"] = X_reduced.columns
vif_data["VIF"] = [variance_inflation_factor(X_reduced.values, i) for i in range(X_reduced.shape[1])]

# Sort by highest VIF
vif_data = vif_data.sort_values(by="VIF", ascending=False)

vif_data.head(20)


,Feature,VIF
38,sen_avg_log,139495.230513
42,h2021_sen_log,120301.359899
19,af_bio_8_x_log,79236.107564
18,af_bio_1_x_log,74445.424706
44,lon_dd_mod_log,49058.131299
21,af_bio_16_x_log,38784.744862
32,anntot_avg_log,36200.478674
34,wetQ_avg_log,33647.216592
41,h2021_grn_log,25725.248242
20,af_bio_13_x_log,19408.503703


In [20]:

from sklearn.preprocessing import StandardScaler
# Separate features (X) and target (y)
X = df_encoded.drop('rainfall', axis=1)  # replace 'rainfall' with your target column name
y = df_encoded['rainfall']

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on training features and transform
X_scaled = scaler.fit_transform(X)

# Convert scaled data back to a DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Combine scaled features with target for final clean dataset
df_scaled = pd.concat([X_scaled_df, y.reset_index(drop=True)], axis=1)

print("Feature scaling completed successfully!")
print("Scaled dataset shape:", df_scaled.shape)
print(df_scaled.head())



Feature scaling completed successfully!
Scaled dataset shape: (514, 80)
   dist_road  dist_market  dist_border  dist_popcenter  dist_admhq  \
0   0.267054     1.690155    -1.339374       -0.978437   -0.975216   
1   0.485348     1.812289    -1.394309       -0.676441   -0.975216   
2  -0.715268     2.453090    -1.693396        1.799178   -0.228180   
3  -0.606121     2.206377    -1.600312       -0.784563   -0.228180   
4  -0.687981    -0.793252    -0.069781       -0.627973    1.265891   

   af_bio_1_x  af_bio_8_x  af_bio_13_x  af_bio_16_x  afmnslp_pct  ...  \
0    1.804142    2.140485    -1.876844    -1.822913    -1.069695  ...   
1    1.827242    2.181956    -1.900197    -1.844561    -1.203375  ...   
2    1.804142    2.389311    -1.993607    -1.887855    -1.069695  ...   
3    1.896542    2.306369    -1.981931    -1.909503    -1.069695  ...   
4   -1.291267   -0.886910     0.890445     0.783422     1.069175  ...   

   ssa_aez09_Tropic-warm/subhumid  landcov_Built-up  \
0            

In [21]:

from sklearn.model_selection import train_test_split

# Split features and target again
X = df_scaled.drop('rainfall', axis=1)
y = df_scaled['rainfall']

# Split into train and test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split successful!")
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


Data split successful!
Training set shape: (411, 79)
Testing set shape: (103, 79)


In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate performance
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(" Model trained successfully!")
print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")


 Model trained successfully!
R² Score: 0.9921
MAE: 23.8327
RMSE: 37.3367


In [23]:
# Check for overfitting
y_train_pred = model.predict(X_train)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_pred)

print(f"R² (Train): {r2_train:.4f}")
print(f"R² (Test): {r2_test:.4f}")



R² (Train): 0.9970
R² (Test): 0.9921


In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Initialize
rf = RandomForestRegressor(n_estimators=200, random_state=42)

# Train
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Metrics
print("R² (Train):", r2_score(y_train, rf.predict(X_train)))
print("R² (Test):", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", mean_squared_error(y_test, y_pred_rf, squared=False))


R² (Train): 0.9999029256547081
R² (Test): 0.9991320780975278
MAE: 7.568792475728161
RMSE: 12.393174405939654


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [25]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Initialize the Decision Tree
dt_model = DecisionTreeRegressor(random_state=42, max_depth=5)  # you can tune max_depth later

# Train
dt_model.fit(X_train, y_train)

# Predict
y_pred = dt_model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Decision Tree Model trained successfully!")
print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

# Train R²
y_train_pred = dt_model.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
print(f"R² (Train): {r2_train:.4f}")


Decision Tree Model trained successfully!
R² Score: 0.9978
MAE: 15.6672
RMSE: 19.8299
R² (Train): 0.9986


In [26]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,      # fewer trees
    max_depth=10,          # limit depth
    max_features='sqrt',   # use subset of features per split
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
print("R² (Train):", r2_score(y_train, rf.predict(X_train)))
print("R² (Test):", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", mean_squared_error(y_test, y_pred_rf, squared=False))


R² (Train): 0.9968521254976009
R² (Test): 0.9789556314749983
MAE: 38.258887609252284
RMSE: 61.025337029840905


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [28]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Train Random Forest on all features
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("=== Full 79-feature model ===")
print(f"R² (Train): {r2_score(y_train, rf.predict(X_train)):.6f}")
print(f"R² (Test): {r2:.6f}")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")


=== Full 79-feature model ===
R² (Train): 0.999903
R² (Test): 0.999132
MAE: 7.568792
RMSE: 12.393174
MSE: 153.590772


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [29]:
from sklearn.inspection import permutation_importance

#  Compute permutation importance
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_importances = pd.Series(perm_result.importances_mean, index=X_train.columns).sort_values(ascending=False)

print("Top 20 features by permutation importance:\n")
print(perm_importances.head(20))

#  Select top 20 features
top_features = perm_importances[:20].index
X_train_reduced = X_train[top_features]
X_test_reduced = X_test[top_features]

#  Retrain RF on top 20 features
rf_reduced = RandomForestRegressor(n_estimators=200, random_state=42)
rf_reduced.fit(X_train_reduced, y_train)


#  Predict and evaluate reduced model
y_pred_reduced = rf_reduced.predict(X_test_reduced)

mse_red = mean_squared_error(y_test, y_pred_reduced)
rmse_red = mean_squared_error(y_test, y_pred_reduced, squared=False)
r2_red = r2_score(y_test, y_pred_reduced)
mae_red = mean_absolute_error(y_test, y_pred_reduced)

print("\n=== Top 20-feature model ===")
print(f"R² (Test): {r2_red:.6f}")
print(f"MAE: {mae_red:.6f}")
print(f"RMSE: {rmse_red:.6f}")
print(f"MSE: {mse_red:.6f}")


Top 20 features by permutation importance:

rainfall_log                  2.086732e+00
af_bio_16_x                   2.421028e-04
af_bio_16_x_log               1.376911e-04
af_bio_13_x_log               2.670378e-05
af_bio_13_x                   1.923424e-05
cropshare                     9.126172e-06
dist_market_log               8.020676e-06
dist_road_log                 7.447697e-06
dist_market                   3.604113e-06
dist_road                     3.167215e-06
twi_ne_log                    3.143950e-06
landcov_Cropland              3.097170e-06
dist_admhq_log                2.492208e-06
anntot_avg_log                2.111833e-06
afmnslp_pct_log               1.912517e-06
wetQ_avg                      1.718342e-06
ssa_aez09_Tropic-warm/arid    1.405593e-06
h2021_wetQstart               9.790345e-07
wetQ_avgstart                 8.907032e-07
dist_admhq                    6.976889e-07
dtype: float64

=== Top 20-feature model ===
R² (Test): 0.999644
MAE: 5.068058
RMSE: 7.941202
MS

C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [30]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np


#  Select top 20 features (from permutation importance)

top_features = perm_importances[:20].index 

X_train_top = X_train[top_features]

# Initialize Random Forest
rf_top = RandomForestRegressor(n_estimators=200, random_state=42)

#  5-fold cross-validation (R²)
cv_scores = cross_val_score(rf_top, X_train_top, y_train, cv=5, scoring='r2')
print("CV R² scores (Top 20 features):", cv_scores)
print("Mean CV R²:", np.mean(cv_scores))

#  Optional: CV MAE
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

y_pred_cv = cross_val_predict(rf_top, X_train_top, y_train, cv=5)
mae_cv = mean_absolute_error(y_train, y_pred_cv)
print("CV MAE (Top 20 features):", mae_cv)


CV R² scores (Top 20 features): [0.99905737 0.99970571 0.99949875 0.99910228 0.99899214]
Mean CV R²: 0.9992712498886369
CV MAE (Top 20 features): 5.923961374695862


In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Select top 20 features
top_features = perm_importances[:20].index  # from permutation importance
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

# Initialize Linear Regression
lr = LinearRegression()
lr.fit(X_train_top, y_train)

# Predict
y_pred_lr = lr.predict(X_test_top)

# Evaluate metrics
mse = mean_squared_error(y_test, y_pred_lr)
rmse = mean_squared_error(y_test, y_pred_lr, squared=False)
r2 = r2_score(y_test, y_pred_lr)
mae = mean_absolute_error(y_test, y_pred_lr)

print("=== Top 20-feature Linear Regression (Train/Test) ===")
print(f"R² (Train): {r2_score(y_train, lr.predict(X_train_top)):.6f}")
print(f"R² (Test): {r2:.6f}")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")


=== Top 20-feature Linear Regression (Train/Test) ===
R² (Train): 0.990272
R² (Test): 0.989483
MAE: 31.714767
RMSE: 43.141445
MSE: 1861.184303


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [32]:
# Select top 20 features from your dataset
top_20_features = [
    "rainfall_log", "af_bio_16_x", "af_bio_16_x_log", "af_bio_13_x_log",
    "af_bio_13_x", "cropshare", "dist_market_log", "dist_road_log",
    "dist_market", "dist_road", "twi_ne_log", "landcov_Cropland",
    "dist_admhq_log", "anntot_avg_log", "afmnslp_pct_log", "wetQ_avg",
    "ssa_aez09_Tropic-warm/arid", "h2021_wetQstart", "wetQ_avgstart",
    "dist_admhq"
]

# Use only these features for training
X_train_top20 = X_train[top_20_features]
X_test_top20 = X_test[top_20_features]

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train_top20, y_train)


RandomForestRegressor(n_estimators=200, random_state=42)

In [33]:
# Example sample (same columns as X_train_top20)
sample_data = {col: [0.5] for col in top_20_features}  # dummy example values
sample_df = pd.DataFrame(sample_data)

# Predict
predicted_rainfall = rf.predict(sample_df)[0]
print("Predicted rainfall:", predicted_rainfall)


Predicted rainfall: 1138.38


In [34]:
actual_rainfall = y_test.iloc[0]  # or the corresponding row
print("Actual rainfall:", actual_rainfall)


Actual rainfall: 1027.0


In [35]:

# New sample data (values are example/scaled or realistic approximate)
sample_data = {
    "rainfall_log": [1000.0],
    "af_bio_16_x": [-1.2],
    "af_bio_16_x_log": [-0.8],
    "af_bio_13_x_log": [0.5],
    "af_bio_13_x": [-1.5],
    "cropshare": [0.3],
    "dist_market_log": [1.5],
    "dist_road_log": [0.2],
    "dist_market": [1.7],
    "dist_road": [0.3],
    "twi_ne_log": [0.4],
    "landcov_Cropland": [-0.8],
    "dist_admhq_log": [-1.0],
    "anntot_avg_log": [0.1],
    "afmnslp_pct_log": [-0.5],
    "wetQ_avg": [0.6],
    "ssa_aez09_Tropic-warm/arid": [-0.3],
    "h2021_wetQstart": [0.0],
    "wetQ_avgstart": [0.2],
    "dist_admhq": [-1.0]
}

# Create DataFrame
new_sample_df = pd.DataFrame(sample_data)

# Predict using your trained Random Forest model
predicted_rainfall = rf.predict(new_sample_df)[0]
print("Predicted rainfall (mm) for new data:", predicted_rainfall)


Predicted rainfall (mm) for new data: 1882.54875


In [36]:
actual_rainfall = 1850.0
print("Predicted rainfall:", predicted_rainfall)
print("Actual rainfall   :", actual_rainfall)
print("Error (Predicted - Actual):", predicted_rainfall - actual_rainfall)


Predicted rainfall: 1882.54875
Actual rainfall   : 1850.0
Error (Predicted - Actual): 32.54874999999993
